# 04 — Demand Forecasting Model: LightGBM & XGBoost
Run **after** `01_Feature_Engineering.ipynb` (and ideally after `03_Baseline_Ridge.ipynb`
so `baseline_summary.csv` exists for the final comparison chart).

**Input:** `forecasting_features.parquet`
**Output:** `model_results.csv`, `model_comparison.csv`

### Target framing: 20-day horizon total
The GA slotting stage needs one demand weight per (Reference, Size) representing expected
activity over the whole evaluation window — not a fresh number every day. So every model in
this project (naive, Ridge, LightGBM, XGBoost) is scored on exactly the same target: **total
picks over the 20 business-day test window.**

### Rolling-origin training snapshots
A single row per combo (its last-known state before the test cutoff) isn't enough to *train* a
model — evaluating on the only row you trained on would leak the answer. So several historical
cutoffs during training are each paired with their own "next 20 business days" total, giving
the model many (features → horizon-total) examples. The **final, untouched cutoff** — the same
one Ridge/naive are scored on — is reserved purely for the final test.

In [ ]:
!pip install lightgbm xgboost

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_style('whitegrid')

panel = pd.read_parquet('forecasting_features.parquet')
panel['unique_id'] = panel['Reference'].astype(str) + "_" + panel['Size (US)'].astype(str)
biz_days = np.sort(panel['date'].unique())
H = 20
final_cutoff_idx = len(biz_days) - H
print("Final cutoff date:", pd.Timestamp(biz_days[final_cutoff_idx]).date())

feature_cols = ['lag_1','lag_2','lag_3','lag_5','lag_10','lag_20',
                 'rmean_5','rmean_10','rmean_20','rstd_5','rstd_10','rstd_20',
                 'freq_10','freq_20','freq_60','nz_mean_20','expanding_mean',
                 'days_since_last','series_age','dow','weekofyear','month','dayofmonth',
                 'ABCCOD_code','Size (US)']  # Sector_code excluded: constant across all products (no predictive value)
print("Number of features:", len(feature_cols))

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# All figures will be saved to a subfolder called "figures" inside the
# directory where you run this notebook.  Change FIGURES_DIR if you prefer
# a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")


## 1. Build rolling-origin training snapshots + final held-out test snapshot

In [ ]:
def build_snapshot(cutoff_idx, panel):
    feat_date = biz_days[cutoff_idx-1]
    target_days = biz_days[cutoff_idx: cutoff_idx+H]
    feats = panel[panel.date==feat_date][['unique_id']+feature_cols]
    tgt = (panel[panel.date.isin(target_days)]
           .groupby('unique_id')['picks'].sum().rename('horizon_total'))
    out = feats.merge(tgt, on='unique_id', how='left')
    out['horizon_total'] = out['horizon_total'].fillna(0)
    return out

train_cutoffs = list(range(40, final_cutoff_idx, H))
print("Rolling training origins:", len(train_cutoffs))

train_data = pd.concat([build_snapshot(c, panel) for c in train_cutoffs], ignore_index=True)
test_data  = build_snapshot(final_cutoff_idx, panel)
Xtr, ytr = train_data[feature_cols], train_data['horizon_total']
Xte, yte = test_data[feature_cols], test_data['horizon_total']
print("Train rows:", Xtr.shape, "| Test rows:", Xte.shape)

## 2. Train LightGBM (Tweedie objective — built for zero-inflated counts)

In [ ]:
lgb_model = lgb.LGBMRegressor(objective='tweedie', tweedie_variance_power=1.3,
                               n_estimators=400, learning_rate=0.05, num_leaves=31,
                               random_state=42, verbosity=-1)
lgb_model.fit(Xtr, ytr)
lgb_pred = np.clip(lgb_model.predict(Xte), 0, None)

lgb_mae  = mean_absolute_error(yte, lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(yte, lgb_pred))
lgb_r2   = r2_score(yte, lgb_pred)
print(f"LightGBM -> MAE {lgb_mae:.3f}  RMSE {lgb_rmse:.3f}  R2 {lgb_r2:.3f}")

## 3. Train XGBoost (Poisson objective)

In [ ]:
xgb_model = xgb.XGBRegressor(objective='count:poisson', n_estimators=400, learning_rate=0.05,
                              max_depth=6, random_state=42, verbosity=0)
xgb_model.fit(Xtr, ytr)
xgb_pred = np.clip(xgb_model.predict(Xte), 0, None)

xgb_mae  = mean_absolute_error(yte, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(yte, xgb_pred))
xgb_r2   = r2_score(yte, xgb_pred)
print(f"XGBoost -> MAE {xgb_mae:.3f}  RMSE {xgb_rmse:.3f}  R2 {xgb_r2:.3f}")

## 4. Skill score vs naive persistence

In [ ]:
prev_days = biz_days[final_cutoff_idx-H: final_cutoff_idx]
naive_pred = (panel[panel.date.isin(prev_days)].groupby('unique_id')['picks'].sum()
              .reindex(test_data['unique_id']).fillna(0).values)
naive_mae = mean_absolute_error(yte, naive_pred)

lgb_skill = 1 - lgb_mae/naive_mae
xgb_skill = 1 - xgb_mae/naive_mae
print(f"Naive MAE: {naive_mae:.3f}")
print(f"LightGBM skill score vs naive: {lgb_skill:.1%}")
print(f"XGBoost  skill score vs naive: {xgb_skill:.1%}")

## 5. Full model comparison table (merges in Ridge/naive from notebook 3 if available)

In [ ]:
rows = [
    {'Model':'LightGBM (Tweedie)', 'MAE':lgb_mae, 'RMSE':lgb_rmse, 'R2':lgb_r2, 'Skill vs naive':lgb_skill},
    {'Model':'XGBoost (Poisson)',  'MAE':xgb_mae, 'RMSE':xgb_rmse, 'R2':xgb_r2, 'Skill vs naive':xgb_skill},
    {'Model':'Naive Persistence',  'MAE':naive_mae, 'RMSE':np.sqrt(mean_squared_error(yte, naive_pred)),
     'R2':r2_score(yte, naive_pred), 'Skill vs naive':0.0},
]
try:
    baseline_summary = pd.read_csv('baseline_summary.csv')
    ridge_row = baseline_summary[baseline_summary.Model=='Ridge Regression'].iloc[0]
    rows.append({'Model':'Ridge Regression', 'MAE':ridge_row.MAE, 'RMSE':ridge_row.RMSE, 'R2':ridge_row.R2,
                 'Skill vs naive': 1 - ridge_row.MAE/naive_mae})
except FileNotFoundError:
    print("baseline_summary.csv not found yet -- run 03_Baseline_Ridge.ipynb to include Ridge in this table.")

comparison = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)
comparison

## 6. Plots — model comparison, predicted vs actual, feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.barplot(x='Model', y='MAE', data=comparison, hue='Model', ax=axes[0], palette='viridis', legend=False)
axes[0].set_title('Model Comparison — MAE (lower is better)')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(x='Model', y='R2', data=comparison, hue='Model', ax=axes[1], palette='viridis', legend=False)
axes[1].set_title('Model Comparison — R² (higher is better)')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "04_model_comparison_mae_r2.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, pred, name, color in zip(axes, [lgb_pred, xgb_pred], ['LightGBM','XGBoost'], ['#4C72B0','#DD8452']):
    ax.scatter(yte, pred, alpha=0.4, s=15, color=color)
    lim = max(yte.max(), pred.max())
    ax.plot([0,lim],[0,lim],'k--', linewidth=1)
    ax.set_xlabel('Actual 20-day total picks'); ax.set_ylabel('Predicted')
    ax.set_title(f'{name}: Predicted vs Actual')
plt.tight_layout(); plt.show()

### Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

lgb_imp = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values()
lgb_imp.tail(15).plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_title('LightGBM — Top 15 Feature Importances')

xgb_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values()
xgb_imp.tail(15).plot(kind='barh', ax=axes[1], color='#DD8452')
axes[1].set_title('XGBoost — Top 15 Feature Importances')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "04_feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Residual analysis (best model)

In [ ]:
best_pred = lgb_pred if lgb_mae <= xgb_mae else xgb_pred
best_name = 'LightGBM' if lgb_mae <= xgb_mae else 'XGBoost'
residuals = yte.values - best_pred

plt.figure(figsize=(8,4))
sns.histplot(residuals, bins=40, color='#55A868')
plt.axvline(0, color='k', linestyle='--')
plt.title(f'{best_name} Residuals (Actual - Predicted)')
plt.xlabel('Residual'); plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "04_residuals.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Save results

In [ ]:
test_out = test_data[['unique_id']].copy()
test_out['actual_total'] = yte.values
test_out['lgb_pred'] = lgb_pred
test_out['xgb_pred'] = xgb_pred
test_out.to_csv('model_results.csv', index=False)
comparison.to_csv('model_comparison.csv', index=False)
print("Saved: model_results.csv, model_comparison.csv")
print("\n'model_results.csv' -> hand this to the GA stage (column 'lgb_pred' = demand weight per combo).")

In [ ]:
# ── Extra graphic: Skill score lollipop chart ─────────────────────────────
import matplotlib.pyplot as plt
import numpy as np, os as _os

models_skill = comparison[comparison['Model'] != 'Naive Persistence'].copy()
models_skill = models_skill.sort_values('Skill vs naive', ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(models_skill)*1.2)))
y_pos = range(len(models_skill))
ax.hlines(y_pos, 0, models_skill['Skill vs naive'], color='#B0BEC5', linewidth=2)
ax.scatter(models_skill['Skill vs naive'], y_pos,
           s=120, color=['#4C72B0' if s > 0 else '#C44E52' for s in models_skill['Skill vs naive']],
           zorder=5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(models_skill['Model'])
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
for i, (val, label) in enumerate(zip(models_skill['Skill vs naive'], models_skill['Model'])):
    ax.text(val + 0.005, i, f"{val:.1%}", va='center', fontsize=10)
ax.set_xlabel('Skill Score vs Naive (higher = better)')
ax.set_title('Forecast Skill Scores vs Naive Persistence', fontweight='bold')
plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "04_skill_score_lollipop.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Extra graphic: All-model RMSE & R2 grouped bar ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#4C72B0','#DD8452','#55A868','#C44E52'][:len(comparison)]

x = np.arange(len(comparison))
w = 0.6
bars0 = axes[0].bar(x, comparison['RMSE'], width=w, color=palette)
axes[0].set_xticks(x); axes[0].set_xticklabels(comparison['Model'], rotation=15, ha='right')
axes[0].set_title('RMSE by Model (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE')
for bar in bars0:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f"{bar.get_height():.1f}", ha='center', va='bottom', fontsize=9)

bars1 = axes[1].bar(x, comparison['R2'], width=w, color=palette)
axes[1].set_xticks(x); axes[1].set_xticklabels(comparison['Model'], rotation=15, ha='right')
axes[1].set_title('R² by Model (higher is better)', fontweight='bold')
axes[1].set_ylabel('R²')
for bar in bars1:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "04_rmse_r2_grouped.png"), dpi=150, bbox_inches="tight")
plt.show()
